# 📊 Notebook 12: Model Evaluation & Comparison

**Objective:** Comprehensive evaluation of all three models (CNN, LSTM, Hybrid CNN-LSTM) with:
- Classification reports
- Confusion matrices
- Model comparison charts
- McNemar's statistical test

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config
from src.preprocessing.sequence_generator import load_sequences
from src.evaluation.metrics import print_model_metrics, compute_metrics
from src.evaluation.confusion_matrix import plot_confusion_matrix
from src.evaluation.model_comparison import compare_models, plot_model_comparison
from src.evaluation.statistical_tests import mcnemar_test

config = load_config('../config/config.yaml')
X_train, X_test, y_train, y_test = load_sequences(
    '../' + config['data']['sequences_path']
)

# Load saved predictions
y_pred_cnn = np.load('../artifacts/y_pred_cnn.npy')
y_pred_lstm = np.load('../artifacts/y_pred_lstm.npy')
y_pred_hybrid = np.load('../artifacts/y_pred_hybrid.npy')

print(f"Test samples: {len(y_test)}")

## 12.1 Individual Model Reports

In [ ]:
cnn_metrics = print_model_metrics('CNN', y_test, y_pred_cnn)
lstm_metrics = print_model_metrics('LSTM', y_test, y_pred_lstm)
hybrid_metrics = print_model_metrics('Hybrid CNN-LSTM', y_test, y_pred_hybrid)

## 12.2 Model Comparison

In [ ]:
results_dict = {
    'CNN': {k: v for k, v in cnn_metrics.items() if k != 'report_dict'},
    'LSTM': {k: v for k, v in lstm_metrics.items() if k != 'report_dict'},
    'Hybrid CNN-LSTM': {k: v for k, v in hybrid_metrics.items() if k != 'report_dict'},
}

comparison_df = compare_models(results_dict, save_path='../results')
plot_model_comparison(comparison_df, save_path='../results/model_comparison.png')

## 12.3 Confusion Matrices (All Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, (name, y_pred) in zip(axes, [
    ('CNN', y_pred_cnn), ('LSTM', y_pred_lstm), ('Hybrid CNN-LSTM', y_pred_hybrid)
]):
    plot_confusion_matrix(y_test, y_pred, name, normalize=True)

plt.tight_layout()
plt.show()

## 12.4 Statistical Tests (McNemar's)

In [ ]:
# CNN vs LSTM
test_1 = mcnemar_test(y_test, y_pred_cnn, y_pred_lstm, 'CNN', 'LSTM')

# CNN vs Hybrid
test_2 = mcnemar_test(y_test, y_pred_cnn, y_pred_hybrid, 'CNN', 'Hybrid CNN-LSTM')

# LSTM vs Hybrid
test_3 = mcnemar_test(y_test, y_pred_lstm, y_pred_hybrid, 'LSTM', 'Hybrid CNN-LSTM')

## 12.5 Final Summary

In [ ]:
print("\n" + "=" * 70)
print("  FINAL MODEL COMPARISON SUMMARY")
print("=" * 70)
print(comparison_df.to_string())
print("\nBest model by F1-Score:", comparison_df['f1_score'].idxmax())
print("=" * 70)

## ✅ Model Evaluation Complete

**Next:** Notebook 13 — Early Warning System